# AI Framework: Quickstart

A guided tour of every layer, rendered with `IPython.display` so model output shows as formatted Markdown, tables, and live-updating streams.

**Before running:**
1. Configure `.env` (copy `.env.example`), then check it with `python examples/01_check_setup.py`.
2. In VS Code, pick the `.venv` kernel (top right of the notebook).
3. Run the cells top to bottom. Sections 5 and 9 also need `EMBEDDING_PROVIDER` / `EMBEDDING_MODEL`.

Each section mirrors a numbered script in `examples/`, which has more detail.

In [ ]:
# Setup: find the project root (works whether the notebook runs from ai-framework/ or notebooks/)
import sys
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "config" / "settings.py").exists()
)
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pandas as pd
from IPython.display import Markdown, display

from aiframe.core import get_llm
from config.settings import ConfigError, get_settings

settings = get_settings()
provider, model = (
    settings.require_llm()
)  # raises a clear ConfigError if .env is incomplete
try:
    settings.require_embeddings()
    HAS_EMBEDDINGS = True
except ConfigError:
    HAS_EMBEDDINGS = False

display(
    Markdown(
        f"**Model:** `{provider} / {model}` · **Embeddings configured:** {HAS_EMBEDDINGS}"
    )
)

## 1. The model factory

`get_llm()` returns a LangChain chat model for whichever provider `.env` names. `display(Markdown(...))` renders the reply as formatted text instead of a raw string.

In [ ]:
llm = get_llm()
response = llm.invoke(
    "Explain retrieval-augmented generation in three short bullet points."
)
display(Markdown(response.content))
response.usage_metadata  # token counts: the basis of cost tracking

## 2. Streaming into one updating cell

`display(..., display_id=True)` returns a handle, and `handle.update(...)` redraws the same output in place. This is the notebook version of a chat UI streaming tokens.

In [ ]:
handle = display(Markdown(""), display_id=True)
text = ""
for chunk in llm.stream(
    "Give three tips for writing a good RAG evaluation set, one line each."
):
    text += chunk.content
    handle.update(Markdown(text))

## 3. Structured output straight into a DataFrame

With `with_structured_output(Schema)`, the model returns validated Pydantic objects, so results drop straight into pandas.

In [ ]:
from aiframe.core.schemas import Answer

structured = get_llm(temperature=0).with_structured_output(Answer)
questions = [
    "What is the capital of Australia?",
    "Who wrote 'The Elements of Statistical Learning'?",
    "What will the S&P 500 close at next Friday?",  # watch the confidence drop
]
pd.DataFrame([structured.invoke(q).model_dump() for q in questions], index=questions)

## 4. Chat with memory

The session replays recent history on every turn. That's why the second question works.

In [ ]:
from aiframe.workflows.chat import ChatSession

session = ChatSession()
for turn in [
    "Hi, I'm Michael. I'm learning MLOps.",
    "What's my name, and what am I learning?",
]:
    display(Markdown(f"**You:** {turn}\n\n**Assistant:** {session.send(turn)}"))

## 5. RAG with hybrid retrieval

Indexes `data/sample/`. First compare what each retriever finds, since most bad answers are retrieval failures, then ask.

In [ ]:
from aiframe.workflows.rag import RAGPipeline

if not HAS_EMBEDDINGS:
    display(
        Markdown(
            "⚠️ Set `EMBEDDING_PROVIDER` and `EMBEDDING_MODEL` in `.env` to run this section."
        )
    )
else:
    rag = RAGPipeline(k=4)
    print(f"Indexed {rag.ingest(ROOT / 'data' / 'sample')} chunks")

    def top_sources(docs):
        return ", ".join(d.metadata["source"] for d in docs[:3])

    queries = ["What does error NX-404 mean?", "How much vacation do new hires get?"]
    display(
        pd.DataFrame(
            {
                "vector": [top_sources(rag.vector_search(q)) for q in queries],
                "keyword (BM25)": [top_sources(rag.keyword_search(q)) for q in queries],
                "hybrid (RRF)": [top_sources(rag.retrieve(q)) for q in queries],
            },
            index=queries,
        )
    )

    result = rag.ask("How many PTO days do new employees get, and how many roll over?")
    display(Markdown(f"{result.answer}\n\n*Sources: {', '.join(result.sources)}*"))

## 6. Tool-calling agent

The agent decides which tools to call. The trace table shows exactly what it did, which is how you debug agents.

In [ ]:
from aiframe.workflows.agent import build_agent, run_agent

agent = build_agent()
result = run_agent(agent, "A team of 12 is on the Pro plan. What is their annual cost?")
display(Markdown(result.answer))
pd.DataFrame([vars(call) for call in result.tool_calls])

## 7. Extraction: text → Pydantic → DataFrame

In [ ]:
from aiframe.core.schemas import ContactList
from aiframe.workflows.extract import extract_many, to_dataframe

emails = [
    "Looping in Dana Reyes (dana.reyes@acme.io), Head of Data at Acme, and Priya Shah, Acme's ML lead.",
    "I'm Tom Becker, procurement manager at Globex. Reach me at tbecker@globex.com.",
]
to_dataframe(extract_many(emails, ContactList), explode="contacts")

## 8. Async batch processing

Jupyter already runs an event loop, so you can `await` directly in a cell; no `asyncio.run` needed.

In [ ]:
from aiframe.workflows.batch import classify_sentiment, summarize
from aiframe.workflows.batch import to_dataframe as batch_to_dataframe

reviews = [
    "Setup took five minutes and the dashboards look great.",
    "Scheduled reports keep timing out. Very frustrating.",
    "It's fine. Does what it says.",
    "Support fixed our issue within an hour. Impressed!",
]
results = await classify_sentiment(reviews, progress=False)
display(summarize(results))
batch_to_dataframe(results)[["input", "label", "score", "reason", "latency_s"]]

## 9. Evaluate against the golden set

Measure before you tune. Note the confidence interval: with about 11 questions it's wide.

In [ ]:
from aiframe.eval import evaluate_rag, load_golden

if HAS_EMBEDDINGS:
    report = evaluate_rag(
        rag, load_golden(ROOT / "data" / "eval" / "golden_set.jsonl"), progress=False
    )
    display(pd.Series(report.summary, name="summary").to_frame())
    display(report.rows[["id", "correct", "judge_score", "hit", "answer"]])

## Next steps

- Change one setting (`k`, `chunk_size`, the prompt version) and rerun section 9 to measure the effect.
- Add your own documents to `data/` and your own questions to `data/eval/golden_set.jsonl`.
- Run the apps: `python apps/gradio_app.py`, `streamlit run apps/streamlit_app.py`, or `uvicorn apps.api:app --reload`.
- Run the tests: `pytest`.